# GNN Assignment 2: Structural Inductive Bias

This notebook compares Feature-Only models (Linear, MLP, DeepSets) with Message Passing models (GCN, GraphSAGE, GAT, GIN) for graph classification. It includes training history visualization (Loss & Metrics).

In [ ]:
# Install dependencies
!pip install torch-geometric ogb

In [ ]:
import os
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch_geometric.loader import DataLoader
from torch_geometric.datasets import TUDataset
from torch_geometric.transforms import OneHotDegree
from torch_geometric.nn import (
    GINConv, GCNConv, SAGEConv, GATv2Conv,
    global_mean_pool
)
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split
from ogb.graphproppred import PygGraphPropPredDataset

## 1. Utilities & Visualization

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def plot_training_history(history, model_name, dataset_name, metric_name):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    ax1.plot(history['train_loss'], label='Train Loss')
    ax1.set_title(f'{model_name} Loss on {dataset_name}')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    
    ax2.plot(history['val_score'], label=f'Val {metric_name}', color='orange')
    ax2.set_title(f'{model_name} {metric_name} on {dataset_name}')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel(metric_name)
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

class EarlyStopping:
    def __init__(self, patience=20, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float('inf')
        self.counter = 0
        self.best_state = None

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
        return self.counter >= self.patience

    def restore_best(self, model):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)

## 2. Models

In [ ]:
def make_atom_encoder(use_atom_encoder, in_channels, hidden_channels):
    if use_atom_encoder:
        from ogb.graphproppred.mol_encoder import AtomEncoder
        return AtomEncoder(emb_dim=hidden_channels)
    return nn.Linear(in_channels, hidden_channels)

def make_mlp(in_dim, hidden_dim, out_dim, num_layers, dropout):
    if num_layers == 1:
        return nn.Linear(in_dim, out_dim)
    layers = [nn.Linear(in_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(), nn.Dropout(dropout)]
    for _ in range(num_layers - 2):
        layers += [nn.Linear(hidden_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(), nn.Dropout(dropout)]
    layers.append(nn.Linear(hidden_dim, out_dim))
    return nn.Sequential(*layers)

### Models
class LinearBaseline(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, use_atom_encoder=True):
        super().__init__()
        self.encoder = make_atom_encoder(use_atom_encoder, in_channels, hidden_channels)
        self.classifier = nn.Linear(hidden_channels, out_channels)
    def forward(self, x, edge_index, batch, **kwargs):
        h = self.encoder(x)
        g = global_mean_pool(h, batch)
        return self.classifier(g)

class MLPBaseline(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=3, dropout=0.5, use_atom_encoder=True):
        super().__init__()
        self.encoder = make_atom_encoder(use_atom_encoder, in_channels, hidden_channels)
        self.mlp = make_mlp(hidden_channels, hidden_channels, out_channels, num_layers, dropout)
    def forward(self, x, edge_index, batch, **kwargs):
        h = self.encoder(x)
        g = global_mean_pool(h, batch)
        return self.mlp(g)

class DeepSets(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=3, dropout=0.5, use_atom_encoder=True):
        super().__init__()
        self.encoder = make_atom_encoder(use_atom_encoder, in_channels, hidden_channels)
        self.phi = make_mlp(hidden_channels, hidden_channels, hidden_channels, num_layers, dropout)
        self.rho = make_mlp(hidden_channels, hidden_channels, out_channels, num_layers, dropout)
    def forward(self, x, edge_index, batch, **kwargs):
        h = self.encoder(x)
        h = self.phi(h)
        g = global_mean_pool(h, batch)
        return self.rho(g)

class GNNModel(nn.Module):
    def __init__(self, model_type, in_channels, hidden_channels, out_channels, num_layers=4, dropout=0.5, use_atom_encoder=True):
        super().__init__()
        self.encoder = make_atom_encoder(use_atom_encoder, in_channels, hidden_channels)
        self.convs, self.bns = nn.ModuleList(), nn.ModuleList()
        self.dropout = dropout
        for _ in range(num_layers):
            if model_type == 'gcn': self.convs.append(GCNConv(hidden_channels, hidden_channels))
            elif model_type == 'sage': self.convs.append(SAGEConv(hidden_channels, hidden_channels))
            elif model_type == 'gat': self.convs.append(GATv2Conv(hidden_channels, hidden_channels, heads=4, concat=False))
            elif model_type == 'gin': 
                mlp = nn.Sequential(nn.Linear(hidden_channels, 2*hidden_channels), nn.BatchNorm1d(2*hidden_channels), nn.ReLU(), nn.Linear(2*hidden_channels, hidden_channels))
                self.convs.append(GINConv(mlp, train_eps=True))
            self.bns.append(nn.BatchNorm1d(hidden_channels))
        self.classifier = nn.Linear(hidden_channels, out_channels)
    def forward(self, x, edge_index, batch, **kwargs):
        h = self.encoder(x)
        for conv, bn in zip(self.convs, self.bns):
            h = conv(h, edge_index); h = bn(h); h = F.relu(h); h = F.dropout(h, p=self.dropout, training=self.training)
        g = global_mean_pool(h, batch)
        return self.classifier(g)

def build_model(model_name, **kwargs):
    name = model_name.lower()
    if name == 'linear': return LinearBaseline(**kwargs)
    if name == 'mlp': return MLPBaseline(**kwargs)
    if name == 'deepsets': return DeepSets(**kwargs)
    return GNNModel(name, **kwargs)

## 3. Data & Training Logic

In [ ]:
class DataManager:
    def __init__(self, name, batch_size=8, seed=42):
        self.name, self.batch_size, self.seed = name, batch_size, seed
    def get_loaders(self):
        if self.name.startswith('ogbg-'):
            dataset = PygGraphPropPredDataset(name=self.name, root='./data')
            split = dataset.get_idx_split()
            return DataLoader(dataset[split['train']], batch_size=self.batch_size, shuffle=True), \
                   DataLoader(dataset[split['valid']], batch_size=self.batch_size), \
                   DataLoader(dataset[split['test']], batch_size=self.batch_size), \
                   {'in': dataset.num_node_features, 'out': dataset.num_tasks, 'metric': 'ROC-AUC', 'encoder': True}
        else:
            dataset = TUDataset(root='./data', name=self.name, use_node_attr=True)
            if dataset._data.x is None: 
                dataset.transform = OneHotDegree(max_degree=100)
                num_feats = 101
            else:
                num_feats = dataset.num_node_features
            indices = list(range(len(dataset)))
            tr_idx, te_idx = train_test_split(indices, test_size=0.1, random_state=self.seed)
            tr_idx, va_idx = train_test_split(tr_idx, test_size=0.1, random_state=self.seed)
            return DataLoader(dataset[tr_idx], batch_size=self.batch_size, shuffle=True), \
                   DataLoader(dataset[va_idx], batch_size=self.batch_size), \
                   DataLoader(dataset[te_idx], batch_size=self.batch_size), \
                   {'in': num_feats, 'out': dataset.num_classes, 'metric': 'Accuracy', 'encoder': False}

def run_train(model, loader, optimizer, device):
    model.train(); total_loss = 0
    for data in loader:
        data = data.to(device); optimizer.zero_grad()
        out = model(data.x, data.edge_index, data.batch)
        y = data.y.view(out.shape).float() if out.shape[1] == 1 else data.y
        loss = F.binary_cross_entropy_with_logits(out, y) if out.shape[1] == 1 else F.cross_entropy(out, y)
        loss.backward(); optimizer.step(); total_loss += loss.item() * data.num_graphs
    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, device, metric_type):
    model.eval(); y_true, y_pred = [], []
    for data in loader:
        data = data.to(device); out = model(data.x, data.edge_index, data.batch)
        y_true.append(data.y.cpu()); y_pred.append(out.cpu())
    y_true, y_pred = torch.cat(y_true, 0).numpy(), torch.cat(y_pred, 0).numpy()
    if metric_type == 'ROC-AUC':
        return roc_auc_score(y_true, y_pred)
    return accuracy_score(y_true, np.argmax(y_pred, 1))

## 4. Experiment Runner

In [ ]:
def run_experiment(model_type, dataset_name, seeds=[42, 43, 44]):
    device = get_device(); scores = []; last_history = None
    
    for seed in seeds:
        set_seed(seed)
        tr_l, va_l, te_l, meta = DataManager(dataset_name, seed=seed).get_loaders()
        model = build_model(model_type, in_channels=meta['in'], hidden_channels=64, out_channels=meta['out'], use_atom_encoder=meta['encoder']).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        stopper = EarlyStopping(patience=20)
        
        history = {'train_loss': [], 'val_score': []}
        for epoch in range(1, 101):
            loss = run_train(model, tr_l, optimizer, device)
            val_score = evaluate(model, va_l, device, meta['metric'])
            history['train_loss'].append(loss)
            history['val_score'].append(val_score)
            if stopper.step(-val_score, model): break
        
        stopper.restore_best(model)
        scores.append(evaluate(model, te_l, device, meta['metric']))
        last_history = history

    print(f"{model_type} on {dataset_name}: {np.mean(scores):.4f} ± {np.std(scores):.4f}")
    plot_training_history(last_history, model_type, dataset_name, meta['metric'])
    return np.mean(scores)

## 5. Main Execution Loop

In [ ]:
# Define experiments to run
datasets = ['MUTAG', 'PROTEINS', 'IMDB-MULTI']
models = ['Linear', 'MLP', 'DeepSets', 'GCN', 'SAGE', 'GAT', 'GIN']

all_results = {}

for data in datasets:
    print(f"\n{'='*20}\nDataset: {data}\n{'='*20}")
    all_results[data] = {}
    for model in models:
        try:
            score = run_experiment(model, data)
            all_results[data][model] = score
        except Exception as e:
            print(f"Error running {model} on {data}: {e}")

print("\nAll experiments completed!")